# Задание 1. RAG по ТФКП

Здесь собраны примеры формул из учебника, ответы Qwen3.5-4B, сравнение вариантов поиска и разбор ошибок. На открытом наборе итоговый вариант выбрал правильные наборы вариантов в 14 из 15 вопросов, baseline — на 12. Значение Q — 0,8986, предварительный балл — 85,32 (при C_ref = 3331,4). Автоматическая метрика проверяет варианты и ссылки; ошибки в обоснованиях ещё остаются. Этот набор использовался при разработке; результат на закрытом тесте пока неизвестен.

Поиск сочетает BM25 и multilingual-e5-small, а mMiniLM заново ранжирует найденные фрагменты. Поиск также использует названия теорем и структуру формул; раздел добавляет вес глобальным кандидатам. В одном запросе Qwen проверяет каждый вариант ответа. Qwen получает шесть основных источников с продолжениями разорванных формулировок одним запросом, в пределах 3500 токенов контекста. Поиск работает на CPU, генерация — через MLX с 4-битными весами на Apple Silicon. В индекс входят все 147 страниц учебника, без вопросов и эталонов.

По умолчанию ячейки показывают сохранённые результаты. Чтобы заново получить ответы, включите `RUN_BEST = True`. Алгоритм находится в Python-модулях; ноутбук запускает его через `run_local.py`.

Для работы с ноутбуком установите зависимости из `requirements/notebook.txt` и выберите это Python-окружение в VS Code или Jupyter. Команды в терминале выполняются из `task-1/`. Подробнее о файлах и запуске — в [README](../README.md).


In [ ]:
import csv
import html
import json
import subprocess
import sys
from pathlib import Path

from IPython.display import HTML, Markdown, display

cwd = Path.cwd().resolve()
ROOT = next(
    (
        path
        for path in (cwd, cwd / "task-1", *cwd.parents)
        if (path / "predict.py").is_file() and (path / "config.json").is_file()
    ),
    None,
)
assert ROOT is not None, "Откройте notebook из репозитория или папки task-1/notebooks"


def rows(name):
    path = ROOT / name
    return (
        [json.loads(s) for s in path.read_text().splitlines() if s.strip()]
        if path.exists()
        else []
    )


def run(*args):
    subprocess.run([sys.executable, *args], cwd=ROOT, check=True)


def escaped(value):
    return html.escape(str(value))


def table(records, columns=None):
    if not records:
        display(Markdown("Результатов пока нет. Сначала выполните нужный этап."))
        return

    columns = columns or list(records[0])
    header = "".join(f"<th>{escaped(column)}</th>" for column in columns)
    body = []
    for record in records:
        cells = "".join(
            f"<td>{escaped(record.get(column, ''))}</td>" for column in columns
        )
        body.append(f"<tr>{cells}</tr>")
    display(
        HTML(
            f"<table><thead><tr>{header}</tr></thead>"
            f"<tbody>{''.join(body)}</tbody></table>"
        )
    )


RUN_BEST = False
print("Задание:", ROOT.name)

## 1. Файлы решения

Корпус извлекается из PDF и сохраняется в `local/data/corpus.jsonl`. Открытые вопросы и эталоны в `datasets/dev/` собраны из выданных QA.

В `dev_answers.jsonl` лежат ответы модели, в `dev_trace.jsonl` — найденные источники и расход токенов. Настройки модели записаны в `runtime.json`. Этот файл подготовлен для локального эксперимента, поскольку готового runtime в материалах не было. Во всех пяти вариантах параметры генерации одинаковые.


In [ ]:
table(
    [
        {"путь": name, "назначение": meaning, "есть": (ROOT / name).exists()}
        for name, meaning in [
            ("run_local.py", "запуск только лучшего RAG"),
            ("build_index.py", "обязательная сборка индекса"),
            ("predict.py", "обязательный CLI предсказания"),
            ("rag/", "внутренние модули"),
            ("scripts/", "подготовка и воспроизведение сравнения"),
            ("materials/", "исходные материалы задания"),
            ("results/", "подробные метрики и ресурсы"),
            ("dev_answers.jsonl", "сохранённые предсказания лучшей системы"),
            ("dev_trace.jsonl", "токены, время, источники"),
            ("experiments.csv", "сравнение и абляции"),
            ("report.md", "выводы и ограничения"),
        ]
    ]
)

## 2. Модели и индекс

Для локального запуска нужны веса Qwen (около 3 ГБ), E5 и mMiniLM (примерно по 470 МБ). Команда `python -m scripts.prepare all --model` скачивает модели и учебник, извлекает корпус и собирает вопросы. Затем индекс строится командой:

```bash
python build_index.py --corpus local/data/corpus.jsonl --output local/index --config config.json
```

Сборка индекса не вызывает LLM. В `local/index/` копируются модели поиска и токенизатор Qwen, поэтому готовый индекс можно использовать отдельно от папки скачанных моделей. При ответах запросы идут только к серверу Qwen.

Индекс формата 3 содержит 2098 поисковых окон для 465 исходных блоков, указатель 18 разделов и названия теорем. Нормализация меняет только поисковые копии. Код автоматически ограничивает длину окна и сохраняет исходные ссылки; сборка не читает QA. Дополнительно из корпуса извлекаются 22 записи признаков формул. Проверена повторная сборка с идентичными векторами.


In [ ]:
table(
    [
        {"компонент": name, "подготовлен": (ROOT / name).exists()}
        for name in [
            "local/data/corpus.jsonl",
            "local/index/index.json",
            "local/models/qwen/model.safetensors",
        ]
    ]
)
print("Если файлов нет, выполните команды подготовки из ячейки выше.")

## 3. Формулы из PDF

При обычном извлечении текста теряются дроби, индексы и обозначения вроде `∂f/∂z̄`. Конвертер использует шрифт, размер и координаты символов, а также линии дробей:

- Кириллица из Type3-шрифтов перекодируется из Latin-1 в CP1251.
- Дроби собираются в `\frac{числитель}{знаменатель}`, индексы — в `_{...}` и `^{...}`.
- Символ `Z` в шрифте CMEX распознаётся как интеграл `\int`.
- Разность множеств записывается через `\setminus`, сопряжение — через `\overline{z}`.

Формула остаётся в одном блоке с окружающим текстом. У блока есть ID для ссылок, номер страницы PDF считается с 1.

Эти правила рассчитаны на конкретный учебник. Многострочные формулы и сложные радикалы могут извлекаться с ошибками. Нераспознанные символы записываются в `local/data/extraction_audit.jsonl` для проверки. LLM при извлечении не используется.


In [ ]:
corpus = rows("local/data/corpus.jsonl")
for page, anchor in [(12, "и мы приходим"), (34, "Теорема 5.3."), (36, "Теорема 6.1.")]:
    block = next(
        (
            block
            for block in corpus
            if block["page"] == page and block["text"].startswith(anchor)
        ),
        None,
    )
    if block:
        formulas = "\n\n".join(
            line
            for line in block["text"].splitlines()
            if line.startswith("$") and "\\frac" in line
        )
        display(Markdown(f"**{block['block_id']} · PDF с. {page}**\n\n" + formulas))
if not corpus:
    print("Для просмотра формул сначала подготовьте локальный corpus.jsonl.")

## 4. Новый прогон

Включите `RUN_BEST = True` в первой ячейке и выполните ноутбук. Скрипт `run_local.py` запустит Qwen, если сервер ещё не работает, обработает 15 вопросов и остановит созданный им сервер. Ответы и trace появятся в `local/outputs/`.

То же самое можно сделать из терминала командой `python run_local.py` с активированным окружением. Параметры генерации берутся из `runtime.json`; thinking отключён. Расход токенов записывается в trace.


In [ ]:
if RUN_BEST:
    run("run_local.py")
else:
    print("Новый прогон отключён; ниже показаны сохранённые результаты.")

## 5. Сравнение вариантов

Проверены пять конфигураций:

1. `no_context` — Qwen без учебника. Оценивается выбор вариантов, Q не рассчитывается.
2. `baseline` — поиск BM25 и один вызов Qwen.
3. `final` — BM25 и E5 по поисковым окнам, mMiniLM, вспомогательный поиск по разделам и приоритет совпадений названий и формул; шесть основных источников с продолжениями и один вызов Qwen.
4. `no_reranker` — итоговый вариант с отключённым mMiniLM.
5. `no_dense` — итоговый вариант с отключённым E5.

Все модели поиска работают на CPU, вопросы обрабатываются независимо. Стоимость по формуле задания: входные токены + 4 × выходные токены + 256 × число вызовов LLM.

Ниже — сохранённые результаты. `RUN_BEST` запускает только итоговый вариант; для повторения всей таблицы используйте команды из [README](../README.md).


In [ ]:
path = ROOT / "experiments.csv"
if path.exists():
    with path.open() as f:
        table(list(csv.DictReader(f)))
else:
    print("Таблица экспериментов отсутствует.")

## 6. Ответы на открытые вопросы

При `RUN_BEST = False` здесь показаны сохранённые ответы из `dev_answers.jsonl`, иначе — результаты нового прогона. Помимо выбранных вариантов, стоит смотреть обоснования и источники: совпадение с эталоном по вариантам ещё не означает, что рассуждение верное.

Обоснование длиннее 1200 символов сокращается до начала и заключения. Полный текст остаётся в поле `model_answer` внутри trace. У явного отказа с пустым набором вариантов очищаются ссылки, как требует формат задания. Математические ошибки эта обработка не исправляет.


In [ ]:
questions = {
    question["id"]: question for question in rows("datasets/dev/dev_questions.jsonl")
}
gold = {reference["id"]: reference for reference in rows("datasets/dev/dev_gold.jsonl")}
answers = rows("local/outputs/answers.jsonl" if RUN_BEST else "dev_answers.jsonl")
traces = {
    trace["id"]: trace
    for trace in rows("local/outputs/trace.jsonl" if RUN_BEST else "dev_trace.jsonl")
}
table(
    [
        {
            "id": answer["id"],
            "выбор": ", ".join(answer["answer_ids"]) or "отказ",
            "эталон": ", ".join(gold[answer["id"]]["answer_ids"]) or "отказ",
            "ссылки": ", ".join(answer["citations"]),
            "токены вход/выход": f"{traces[answer['id']]['input_tokens']} / {traces[answer['id']]['output_tokens']}",
        }
        for answer in answers
    ]
)
for answer in answers:
    question = questions[answer["id"]]
    display(
        HTML(
            "<details><summary>"
            + html.escape(answer["id"] + ": " + question["question"])
            + "</summary><p>"
            + html.escape(answer["rationale"])
            + "</p><p>Источники: "
            + html.escape(", ".join(answer["citations"]))
            + "</p></details>"
        )
    )

## 7. Ошибки сохранённого прогона

В таблицу попадают неверно выбранные варианты и ответы с неполным совпадением ссылок. Для нескольких вопросов ниже показаны фрагменты, которые нашёл поиск.

Эталоны ссылок составлены по исходным QA. Они могут не включать все допустимые доказательства, поэтому низкий F1 ссылок нужно проверять по тексту источников.


In [ ]:
path = ROOT / "results/experiment_details.json"
if path.exists():
    results = json.loads(path.read_text())
    saved_traces = {trace["id"]: trace for trace in rows("dev_trace.jsonl")}
    errors = [
        row
        for row in results["final"]["details"]
        if row["A"] == 0 or (not gold[row["id"]]["abstain"] and row["F"] < 1)
    ]
    table(errors, ["id", "category", "A", "F", "score", "error"])
    by_block = {block["block_id"]: block for block in corpus}
    for error in errors[:3]:
        question_id = error["id"]
        display(
            Markdown("**" + question_id + "**: " + questions[question_id]["question"])
        )
        table(
            [
                {
                    "block_id": block_id,
                    "page": by_block.get(block_id, {}).get("page", ""),
                    "фрагмент": by_block.get(block_id, {}).get("text", "")[:90],
                }
                for block_id in saved_traces[question_id]["retrieved_block_ids"]
            ]
        )
    table(
        [
            {
                "id": trace["id"],
                "найдено блоков": len(trace["retrieved_block_ids"]),
                "вызовов": trace["llm_calls"],
            }
            for trace in saved_traces.values()
        ]
    )

## 8. Авторские вопросы и ресурсы

Восемь авторских вопросов охватывают четыре категории задания. После них приведены время сборки индекса, его размер и расход памяти из сохранённого отчёта.

Дополнительная проверка дала 5/8 правильных наборов вариантов (Q=0,5250); у непосредственно предыдущей версии индекса было 4/8. Исправлены own_001 и own_003, но ухудшился own_002. Этого малого набора недостаточно для выводов о закрытом тесте. Подробности — в `results/own_metrics.json`.


In [ ]:
own_gold = {reference["id"]: reference for reference in rows("own_gold.jsonl")}
table(
    [
        {
            "id": question["id"],
            "категория": own_gold[question["id"]]["category"],
            "вопрос": question["question"],
            "эталон": own_gold[question["id"]]["answer_ids"] or "отказ",
        }
        for question in rows("own_questions.jsonl")
    ]
)
path = ROOT / "results/build_stats.json"
if path.exists():
    table([json.loads(path.read_text())])

## 9. Файлы для сдачи

В репозиторий входят код, ноутбук, зависимости, настройки, вопросы с эталонами, отчёт и результаты экспериментов. Папка `local/`, окружение, Python-кэш, логи и `.env` исключены через `.gitignore`.

Перед сдачей стоит прочитать [отчёт](../report.md), выполнить `python -m unittest discover -s tests` и проверить команды из README с заново собранным индексом.
